TRYING CANINE PLUS CONTRASTIVE PLUS PROTO DAR CU OOS RECALL

In [1]:
!pip install transformers torch nlpaug -q

from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, torch.nn.functional as F
import json, random, numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import CanineTokenizer, CanineModel, get_linear_schedule_with_warmup
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
from sklearn.metrics import accuracy_score
import os

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR  = "/content/drive/MyDrive/NLP/bert-clinc/canine_151"
MAX_LEN   = 1024
OOS_ID    = 150
NUM_CLASS = 151

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 30.8 MB/s eta 0:00:00
Mounted at /content/drive
Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [2]:
with open("/content/drive/MyDrive/NLP/oos-eval/data/data_full.json") as f:
    data = json.load(f)

# In-scope
train_texts  = [x[0] for x in data["train"]]
train_labels = [x[1] for x in data["train"]]
val_texts    = [x[0] for x in data["val"]]
val_labels   = [x[1] for x in data["val"]]
test_texts   = [x[0] for x in data["test"]]
test_labels  = [x[1] for x in data["test"]]

# OOS
oos_train_texts = [x[0] for x in data["oos_train"]]
oos_val_texts   = [x[0] for x in data["oos_val"]]
oos_test_texts  = [x[0] for x in data["oos_test"]]

# Label mapping
in_scope  = sorted(set(train_labels))
label2id  = {l: i for i, l in enumerate(in_scope)}
label2id["oos"] = OOS_ID

# Combine in-scope + OOS
all_train_texts  = train_texts  + oos_train_texts
all_train_labels = [label2id[l] for l in train_labels] + [OOS_ID] * len(oos_train_texts)
all_val_texts    = val_texts    + oos_val_texts
all_val_labels   = [label2id[l] for l in val_labels]   + [OOS_ID] * len(oos_val_texts)

# Keep separate for evaluation
inscope_test_ids = [label2id[l] for l in test_labels]
oos_test_ids     = [OOS_ID] * len(oos_test_texts)

tokenizer = CanineTokenizer.from_pretrained("google/canine-s")

print(f"Train: {len(all_train_texts)} | Val: {len(all_val_texts)}")
print(f"In-scope test: {len(test_texts)} | OOS test: {len(oos_test_texts)}")
print(f"Total classes: {NUM_CLASS}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/854 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/657 [00:00<?, ?B/s]

Train: 15100 | Val: 3100
In-scope test: 4500 | OOS test: 1000
Total classes: 151


In [7]:
class SimpleDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts   = texts
        self.labels  = labels
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], truncation=True,
                       padding="max_length", max_length=self.max_len,
                       return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label":          torch.tensor(self.labels[idx])
        }

class ContrastiveDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts   = texts
        self.labels  = labels
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        def enc(t):
            return self.tok(t, truncation=True, padding="max_length",
                            max_length=self.max_len, return_tensors="pt")
        c = enc(self.texts[idx])
        n = enc(generate_noisy(self.texts[idx]))
        return {
            "clean_input_ids":      c["input_ids"].squeeze(),
            "clean_attention_mask": c["attention_mask"].squeeze(),
            "noisy_input_ids":      n["input_ids"].squeeze(),
            "noisy_attention_mask": n["attention_mask"].squeeze(),
            "label": torch.tensor(self.labels[idx])
        }

class CanineIntent(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder    = CanineModel.from_pretrained("google/canine-s")
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, num_classes)
        self.prototypes = None

    def get_embedding(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.dropout(out.last_hidden_state[:, 0, :])

    def forward(self, input_ids, attention_mask, labels=None):
        emb    = self.get_embedding(input_ids, attention_mask)
        logits = self.classifier(emb)
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            return loss, logits
        return logits

# Noise functions
kb_aug = nac.KeyboardAug(aug_char_p=0.15)
sp_aug = naw.SpellingAug(aug_p=0.2)

def generate_noisy(text):
    choice = random.choices(["keyboard", "spelling"], weights=[0.7, 0.3])[0]
    try:
        return kb_aug.augment(text)[0] if choice == "keyboard" else sp_aug.augment(text)[0]
    except:
        return text

def compute_metrics_linear(model, texts, label_ids, batch_size=64):
    model.eval()
    preds, labs = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(batch, truncation=True, padding=True,
                          max_length=MAX_LEN, return_tensors="pt")
        with torch.no_grad():
            emb    = model.get_embedding(enc["input_ids"].to(device),
                                          enc["attention_mask"].to(device))
            logits = model.classifier(emb)
            pred   = torch.argmax(logits, dim=-1).cpu().numpy()
        preds.extend(pred)
        labs.extend(label_ids[i:i+batch_size])

    preds = np.array(preds)
    labs  = np.array(labs)
    acc   = accuracy_score(labs, preds)
    oos_mask   = labs == OOS_ID
    oos_recall = (preds[oos_mask] == OOS_ID).mean() if oos_mask.any() else 0.0
    return acc, oos_recall

def compute_metrics_proto(model, texts, label_ids, batch_size=32):
    model.eval()
    preds, labs = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(batch, truncation=True, padding=True,
                          max_length=MAX_LEN, return_tensors="pt")
        with torch.no_grad():
            emb  = model.get_embedding(enc["input_ids"].to(device),
                                        enc["attention_mask"].to(device))
            dist = torch.cdist(emb, model.prototypes)
            pred = torch.argmin(dist, dim=-1).cpu().numpy()
        preds.extend(pred)
        labs.extend(label_ids[i:i+batch_size])

    preds = np.array(preds)
    labs  = np.array(labs)
    acc   = accuracy_score(labs, preds)
    oos_mask   = labs == OOS_ID
    oos_recall = (preds[oos_mask] == OOS_ID).mean() if oos_mask.any() else 0.0
    return acc, oos_recall

def print_results(model, label="", use_proto=False):
    noisy_variants = {
        "original":      test_texts,
        "casing":        [t.upper() for t in test_texts],
        "keyboard":      [kb_aug.augment(t)[0] if True else t for t in test_texts],
        "spelling":      [sp_aug.augment(t)[0] if True else t for t in test_texts],
        "synonyms":      test_texts,  # skip for speed, same as original for CANINE
        "abbreviations": test_texts,  # same
    }
    fn = compute_metrics_proto if use_proto else compute_metrics_linear

    print(f"\n{'─'*65}")
    print(f"Results: {label}")
    print(f"{'─'*65}")
    print(f"{'Noise':<20} {'In-scope Acc':>14} {'OOS Recall':>12}")
    print(f"{'─'*65}")
    for noise_type, texts in noisy_variants.items():
        acc, oos_r = fn(model, texts, inscope_test_ids)
        print(f"{noise_type:<20} {acc*100:>13.2f}% {oos_r*100:>11.2f}%")

    # OOS test recall
    _, oos_r = fn(model, oos_test_texts, oos_test_ids)
    print(f"{'─'*65}")
    print(f"{'OOS test recall':<20} {'—':>14} {oos_r*100:>11.2f}%")

print("All functions ready")

All functions ready


In [8]:
model = CanineIntent(num_classes=NUM_CLASS).to(device)

train_dataset = SimpleDataset(all_train_texts, all_train_labels, tokenizer, MAX_LEN)
val_dataset   = SimpleDataset(all_val_texts,   all_val_labels,   tokenizer, MAX_LEN)
train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=32)

EPOCHS    = 5
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=len(train_loader) * EPOCHS)

print("Starting clean CANINE training (151 classes)...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        ids    = batch["input_ids"].to(device)
        mask   = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad()
        loss, _ = model(ids, mask, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if step % 200 == 0:
            print(f"Epoch {epoch+1} Step {step}/{len(train_loader)} Loss: {loss.item():.4f}")

    # Quick val accuracy
    model.eval()
    val_preds, val_labs = [], []
    with torch.no_grad():
        for batch in val_loader:
            ids    = batch["input_ids"].to(device)
            mask   = batch["attention_mask"].to(device)
            logits = model(ids, mask)
            val_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            val_labs.extend(batch["label"].numpy())
    val_acc = accuracy_score(val_labs, val_preds)

    print(f"Epoch {epoch+1} done — Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc*100:.2f}%")
    torch.save(model.state_dict(), f"{SAVE_DIR}/canine_clean_epoch{epoch+1}.pt")
    print(f"Saved epoch {epoch+1}")

Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

CanineModel LOAD REPORT from: google/canine-s
Key                          | Status     |  | 
-----------------------------+------------+--+-
char_embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting clean CANINE training (151 classes)...
Epoch 1 Step 0/944 Loss: 5.0928
Epoch 1 Step 200/944 Loss: 5.0447
Epoch 1 Step 400/944 Loss: 5.0296
Epoch 1 Step 600/944 Loss: 5.1427
Epoch 1 Step 800/944 Loss: 4.7263
Epoch 1 done — Loss: 4.8689 | Val Acc: 18.10%
Saved epoch 1
Epoch 2 Step 0/944 Loss: 3.6592
Epoch 2 Step 200/944 Loss: 2.2002
Epoch 2 Step 400/944 Loss: 2.1719
Epoch 2 Step 600/944 Loss: 1.3155
Epoch 2 Step 800/944 Loss: 1.2420
Epoch 2 done — Loss: 2.1136 | Val Acc: 76.26%
Saved epoch 2
Epoch 3 Step 0/944 Loss: 0.7815
Epoch 3 Step 200/944 Loss: 0.4651
Epoch 3 Step 400/944 Loss: 0.8885
Epoch 3 Step 600/944 Loss: 0.6242
Epoch 3 Step 800/944 Loss: 0.5218
Epoch 3 done — Loss: 0.7207 | Val Acc: 84.03%
Saved epoch 3
Epoch 4 Step 0/944 Loss: 0.5175
Epoch 4 Step 200/944 Loss: 0.4669
Epoch 4 Step 400/944 Loss: 0.4568
Epoch 4 Step 600/944 Loss: 0.0462
Epoch 4 Step 800/944 Loss: 0.6552
Epoch 4 done — Loss: 0.4170 | Val Acc: 87.10%
Saved epoch 4
Epoch 5 Step 0/944 Loss: 0.4171
Epoch 5 

In [9]:
import os
files = os.listdir(f"{SAVE_DIR}")
print(files)

['canine_clean_epoch1.pt', 'canine_clean_epoch2.pt', 'canine_clean_epoch3.pt', 'canine_clean_epoch4.pt', 'canine_clean_epoch5.pt']


In [10]:
model = CanineIntent(num_classes=NUM_CLASS).to(device)
model.load_state_dict(torch.load(f"{SAVE_DIR}/canine_clean_epoch5.pt", map_location=device))
model.eval()
print("Model loaded")
print_results(model, label="CANINE Clean (151 classes)", use_proto=False)

Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

CanineModel LOAD REPORT from: google/canine-s
Key                          | Status     |  | 
-----------------------------+------------+--+-
char_embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded

─────────────────────────────────────────────────────────────────
Results: CANINE Clean (151 classes)
─────────────────────────────────────────────────────────────────
Noise                  In-scope Acc   OOS Recall
─────────────────────────────────────────────────────────────────
original                     89.42%        0.00%
casing                       75.91%        0.00%
keyboard                     71.16%        0.00%
spelling                     84.67%        0.00%
synonyms                     89.42%        0.00%
abbreviations                89.42%        0.00%
─────────────────────────────────────────────────────────────────
OOS test recall                   —       18.30%


In [11]:
# Load best clean checkpoint if new session:
# model = CanineIntent(num_classes=NUM_CLASS).to(device)
# model.load_state_dict(torch.load(f"{SAVE_DIR}/canine_clean_epoch5.pt"))

# Contrastive only on in-scope — noisy OOS pairs don't make semantic sense
inscope_train_ids = [label2id[l] for l in train_labels]
cont_dataset = ContrastiveDataset(train_texts, inscope_train_ids, tokenizer, MAX_LEN)
cont_loader  = DataLoader(cont_dataset, batch_size=8, shuffle=True)

CONT_EPOCHS = 3
optimizer   = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler   = get_linear_schedule_with_warmup(optimizer,
    num_warmup_steps=len(cont_loader),
    num_training_steps=len(cont_loader) * CONT_EPOCHS)

print("Starting contrastive fine-tuning...")
for epoch in range(CONT_EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(cont_loader):
        c_ids  = batch["clean_input_ids"].to(device)
        c_mask = batch["clean_attention_mask"].to(device)
        n_ids  = batch["noisy_input_ids"].to(device)
        n_mask = batch["noisy_attention_mask"].to(device)
        labels = batch["label"].to(device)

        clean_emb = model.get_embedding(c_ids, c_mask)
        noisy_emb = model.get_embedding(n_ids, n_mask)
        logits    = model.classifier(clean_emb)

        ce_loss = nn.CrossEntropyLoss()(logits, labels)
        cn      = F.normalize(clean_emb, dim=-1)
        nn_     = F.normalize(noisy_emb, dim=-1)
        B       = cn.size(0)
        embs    = torch.cat([cn, nn_], dim=0)
        sim     = torch.matmul(embs, embs.T) / 0.07
        mask_   = torch.eye(2*B, device=embs.device).bool()
        sim.masked_fill_(mask_, float('-inf'))
        cl_labels = torch.cat([torch.arange(B, 2*B), torch.arange(B)]).to(embs.device)
        cl_loss   = nn.CrossEntropyLoss()(sim, cl_labels)

        loss = ce_loss + 0.3 * cl_loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if step % 200 == 0:
            print(f"Epoch {epoch+1} Step {step}/{len(cont_loader)} "
                  f"Loss: {loss.item():.4f} CE: {ce_loss.item():.4f} CL: {cl_loss.item():.4f}")

    print(f"Epoch {epoch+1} done — Avg Loss: {total_loss/len(cont_loader):.4f}")
    torch.save(model.state_dict(), f"{SAVE_DIR}/canine_contrastive_epoch{epoch+1}.pt")
    print(f"Saved contrastive epoch {epoch+1}")

Starting contrastive fine-tuning...
Epoch 1 Step 0/1875 Loss: 0.1093 CE: 0.0812 CL: 0.0938
Epoch 1 Step 200/1875 Loss: 0.4767 CE: 0.2462 CL: 0.7682
Epoch 1 Step 400/1875 Loss: 0.3998 CE: 0.3064 CL: 0.3113
Epoch 1 Step 600/1875 Loss: 0.9899 CE: 0.9566 CL: 0.1108
Epoch 1 Step 800/1875 Loss: 0.3656 CE: 0.3333 CL: 0.1077
Epoch 1 Step 1000/1875 Loss: 0.6263 CE: 0.6004 CL: 0.0864
Epoch 1 Step 1200/1875 Loss: 0.6874 CE: 0.6672 CL: 0.0673
Epoch 1 Step 1400/1875 Loss: 0.7749 CE: 0.7585 CL: 0.0547
Epoch 1 Step 1600/1875 Loss: 0.4550 CE: 0.4364 CL: 0.0619
Epoch 1 Step 1800/1875 Loss: 0.2626 CE: 0.2558 CL: 0.0227
Epoch 1 done — Avg Loss: 0.3043
Saved contrastive epoch 1
Epoch 2 Step 0/1875 Loss: 0.0597 CE: 0.0334 CL: 0.0877
Epoch 2 Step 200/1875 Loss: 0.0543 CE: 0.0443 CL: 0.0334
Epoch 2 Step 400/1875 Loss: 0.1962 CE: 0.1528 CL: 0.1448
Epoch 2 Step 600/1875 Loss: 0.0433 CE: 0.0251 CL: 0.0604
Epoch 2 Step 800/1875 Loss: 0.0854 CE: 0.0410 CL: 0.1479
Epoch 2 Step 1000/1875 Loss: 0.0216 CE: 0.0156 CL:

In [12]:
print_results(model, label="CANINE + Contrastive (151 classes)", use_proto=False)


─────────────────────────────────────────────────────────────────
Results: CANINE + Contrastive (151 classes)
─────────────────────────────────────────────────────────────────
Noise                  In-scope Acc   OOS Recall
─────────────────────────────────────────────────────────────────
original                     91.76%        0.00%
casing                       80.60%        0.00%
keyboard                     84.89%        0.00%
spelling                     88.89%        0.00%
synonyms                     91.76%        0.00%
abbreviations                91.76%        0.00%
─────────────────────────────────────────────────────────────────
OOS test recall                   —        0.00%


In [13]:
# Load if new session:
# model = CanineIntent(num_classes=NUM_CLASS).to(device)
# model.load_state_dict(torch.load(f"{SAVE_DIR}/canine_contrastive_epoch3.pt"))

print("Building prototypes...")
model.eval()
embs_per_class = {i: [] for i in range(NUM_CLASS)}

# In-scope prototypes
for i, (text, label) in enumerate(zip(train_texts, inscope_train_ids)):
    enc = tokenizer(text, truncation=True, padding="max_length",
                    max_length=MAX_LEN, return_tensors="pt")
    with torch.no_grad():
        emb = model.get_embedding(enc["input_ids"].to(device),
                                   enc["attention_mask"].to(device))
    embs_per_class[label].append(emb.squeeze().cpu())
    if i % 2000 == 0:
        print(f"  In-scope: {i}/{len(train_texts)}")

# OOS prototype — built from oos_train examples
print("Building OOS prototype...")
for i, text in enumerate(oos_train_texts):
    enc = tokenizer(text, truncation=True, padding="max_length",
                    max_length=MAX_LEN, return_tensors="pt")
    with torch.no_grad():
        emb = model.get_embedding(enc["input_ids"].to(device),
                                   enc["attention_mask"].to(device))
    embs_per_class[OOS_ID].append(emb.squeeze().cpu())

prototypes = torch.stack([
    torch.stack(embs_per_class[i]).mean(dim=0)
    for i in range(NUM_CLASS)
]).to(device)

model.prototypes = prototypes
torch.save(prototypes, f"{SAVE_DIR}/prototypes.pt")
print(f"Prototypes built — shape: {prototypes.shape}")

Building prototypes...
  In-scope: 0/15000
  In-scope: 2000/15000
  In-scope: 4000/15000
  In-scope: 6000/15000
  In-scope: 8000/15000
  In-scope: 10000/15000
  In-scope: 12000/15000
  In-scope: 14000/15000
Building OOS prototype...
Prototypes built — shape: torch.Size([151, 768])


In [14]:
print_results(model, label="CANINE + Contrastive + Prototypical (151 classes)", use_proto=True)


─────────────────────────────────────────────────────────────────
Results: CANINE + Contrastive + Prototypical (151 classes)
─────────────────────────────────────────────────────────────────
Noise                  In-scope Acc   OOS Recall
─────────────────────────────────────────────────────────────────
original                     90.71%        0.00%
casing                       58.20%        0.00%
keyboard                     81.96%        0.00%
spelling                     86.84%        0.00%
synonyms                     90.71%        0.00%
abbreviations                90.71%        0.00%
─────────────────────────────────────────────────────────────────
OOS test recall                   —       39.90%


In [15]:
print("\n" + "═"*65)
print("FULL COMPARISON — CANINE 151 classes")
print("═"*65)

configs = [
    (f"{SAVE_DIR}/canine_clean_epoch5.pt",          "CANINE Clean",                False),
    (f"{SAVE_DIR}/canine_contrastive_epoch3.pt",    "CANINE + Contrastive",        False),
    (f"{SAVE_DIR}/canine_contrastive_epoch3.pt",    "CANINE + Contrastive + Proto",True),
]

for path, label, use_proto in configs:
    m = CanineIntent(num_classes=NUM_CLASS).to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    if use_proto:
        m.prototypes = torch.load(f"{SAVE_DIR}/prototypes.pt", map_location=device)
    m.eval()

    fn = compute_metrics_proto if use_proto else compute_metrics_linear
    acc_clean, _      = fn(m, test_texts,     inscope_test_ids)
    acc_keyboard, _   = fn(m, [kb_aug.augment(t)[0] for t in test_texts], inscope_test_ids)
    _, oos_r          = fn(m, oos_test_texts, oos_test_ids)

    print(f"\n{label}")
    print(f"  Clean:    {acc_clean*100:.2f}%")
    print(f"  Keyboard: {acc_keyboard*100:.2f}%")
    print(f"  OOS Recall: {oos_r*100:.2f}%")


═════════════════════════════════════════════════════════════════
FULL COMPARISON — CANINE 151 classes
═════════════════════════════════════════════════════════════════


Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

CanineModel LOAD REPORT from: google/canine-s
Key                          | Status     |  | 
-----------------------------+------------+--+-
char_embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



CANINE Clean
  Clean:    89.42%
  Keyboard: 70.67%
  OOS Recall: 18.30%


Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

CanineModel LOAD REPORT from: google/canine-s
Key                          | Status     |  | 
-----------------------------+------------+--+-
char_embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



CANINE + Contrastive
  Clean:    91.76%
  Keyboard: 84.76%
  OOS Recall: 0.00%


Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

CanineModel LOAD REPORT from: google/canine-s
Key                          | Status     |  | 
-----------------------------+------------+--+-
char_embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



CANINE + Contrastive + Proto
  Clean:    90.71%
  Keyboard: 81.69%
  OOS Recall: 39.90%


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
print(os.listdir("/content/drive/MyDrive/NLP/bert-clinc/canine_151"))

['canine_clean_epoch1.pt', 'canine_clean_epoch2.pt', 'canine_clean_epoch3.pt', 'canine_clean_epoch4.pt', 'canine_clean_epoch5.pt', 'canine_contrastive_epoch1.pt', 'canine_contrastive_epoch2.pt', 'canine_contrastive_epoch3.pt', 'prototypes.pt']


In [4]:
!pip install transformers torch nlpaug -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 26.7 MB/s eta 0:00:00


In [5]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import numpy as np
from transformers import CanineTokenizer, CanineModel
from sklearn.metrics import accuracy_score
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw

# ── Config ─────────────────────────────────────────────
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR  = "/content/drive/MyDrive/NLP/bert-clinc/canine_151"
MAX_LEN   = 1024
OOS_ID    = 150
NUM_CLASS = 151

# ── Load data ──────────────────────────────────────────
with open("/content/drive/MyDrive/NLP/oos-eval/data/data_full.json") as f:
    data = json.load(f)

test_texts     = [x[0] for x in data["test"]]
test_labels    = [x[1] for x in data["test"]]
oos_test_texts = [x[0] for x in data["oos_test"]]
train_labels   = [x[1] for x in data["train"]]

in_scope = sorted(set(train_labels))
label2id = {l: i for i, l in enumerate(in_scope)}
label2id["oos"] = OOS_ID

inscope_test_ids = [label2id[l] for l in test_labels]
oos_test_ids     = [OOS_ID] * len(oos_test_texts)

tokenizer = CanineTokenizer.from_pretrained("google/canine-s")

# ── Model ──────────────────────────────────────────────
class CanineIntent(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder    = CanineModel.from_pretrained("google/canine-s")
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, num_classes)
        self.prototypes = None

    def get_embedding(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.dropout(out.last_hidden_state[:, 0, :])

model = CanineIntent(num_classes=NUM_CLASS).to(device)
model.load_state_dict(torch.load(f"{SAVE_DIR}/canine_contrastive_epoch3.pt",
                                  map_location=device))
model.prototypes = torch.load(f"{SAVE_DIR}/prototypes.pt", map_location=device)
model.eval()
print("Model and prototypes loaded")

# ── Noise functions ────────────────────────────────────
kb_aug = nac.KeyboardAug(aug_char_p=0.15)
sp_aug = naw.SpellingAug(aug_p=0.2)

ABBREV_MAP = {
    "please": "pls", "you": "u", "your": "ur", "are": "r",
    "okay": "ok", "thanks": "thx", "tomorrow": "tmrw", "because": "bc",
    "with": "w/", "without": "w/o", "information": "info",
    "application": "app", "number": "num", "message": "msg",
    "account": "acct", "password": "pwd", "transfer": "trnsfr",
}

def apply_noise(texts, noise_type):
    if noise_type == "original":
        return texts
    elif noise_type == "casing":
        return [t.upper() for t in texts]
    elif noise_type == "keyboard":
        out = []
        for t in texts:
            try: out.append(kb_aug.augment(t)[0])
            except: out.append(t)
        return out
    elif noise_type == "spelling":
        out = []
        for t in texts:
            try: out.append(sp_aug.augment(t)[0])
            except: out.append(t)
        return out
    elif noise_type == "abbreviations":
        out = []
        for t in texts:
            words = t.split()
            out.append(" ".join(ABBREV_MAP.get(w.lower(), w) for w in words))
        return out
    return texts

# ── Eval functions ─────────────────────────────────────
def eval_linear(texts, label_ids, batch_size=64):
    preds, labs = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(batch, truncation=True, padding=True,
                          max_length=MAX_LEN, return_tensors="pt")
        with torch.no_grad():
            emb    = model.get_embedding(enc["input_ids"].to(device),
                                          enc["attention_mask"].to(device))
            logits = model.classifier(emb)
            pred   = torch.argmax(logits, dim=-1).cpu().numpy()
        preds.extend(pred)
        labs.extend(label_ids[i:i+batch_size])
    preds = np.array(preds)
    labs  = np.array(labs)
    acc   = accuracy_score(labs, preds)
    oos_mask   = labs == OOS_ID
    oos_recall = (preds[oos_mask] == OOS_ID).mean() if oos_mask.any() else 0.0
    return acc, oos_recall

def eval_proto(texts, label_ids, batch_size=32):
    preds, labs = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(batch, truncation=True, padding=True,
                          max_length=MAX_LEN, return_tensors="pt")
        with torch.no_grad():
            emb  = model.get_embedding(enc["input_ids"].to(device),
                                        enc["attention_mask"].to(device))
            dist = torch.cdist(emb, model.prototypes)
            pred = torch.argmin(dist, dim=-1).cpu().numpy()
        preds.extend(pred)
        labs.extend(label_ids[i:i+batch_size])
    preds = np.array(preds)
    labs  = np.array(labs)
    acc   = accuracy_score(labs, preds)
    oos_mask   = labs == OOS_ID
    oos_recall = (preds[oos_mask] == OOS_ID).mean() if oos_mask.any() else 0.0
    return acc, oos_recall

# ── Full evaluation ────────────────────────────────────
noise_types = ["original", "casing", "keyboard", "spelling", "abbreviations"]

print(f"\n{'─'*70}")
print("In-scope Accuracy per Noise Type")
print(f"{'─'*70}")
print(f"{'Noise':<20} {'Linear Acc':>12} {'Proto Acc':>12}")
print(f"{'─'*70}")
for nt in noise_types:
    noisy = apply_noise(test_texts, nt)
    acc_l, _ = eval_linear(noisy, inscope_test_ids)
    acc_p, _ = eval_proto(noisy, inscope_test_ids)
    print(f"{nt:<20} {acc_l*100:>11.2f}% {acc_p*100:>11.2f}%")

print(f"\n{'─'*70}")
print("OOS Recall per Noise Type (on OOS test queries)")
print(f"{'─'*70}")
print(f"{'Noise':<20} {'Linear OOS':>12} {'Proto OOS':>12}")
print(f"{'─'*70}")
for nt in noise_types:
    noisy_oos    = apply_noise(oos_test_texts, nt)
    _, oos_l     = eval_linear(noisy_oos, oos_test_ids)
    _, oos_p     = eval_proto(noisy_oos, oos_test_ids)
    print(f"{nt:<20} {oos_l*100:>11.2f}% {oos_p*100:>11.2f}%")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/854 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/657 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/528M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

CanineModel LOAD REPORT from: google/canine-s
Key                          | Status     |  | 
-----------------------------+------------+--+-
char_embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model and prototypes loaded

──────────────────────────────────────────────────────────────────────
In-scope Accuracy per Noise Type
──────────────────────────────────────────────────────────────────────
Noise                  Linear Acc    Proto Acc
──────────────────────────────────────────────────────────────────────
original                   91.76%       90.71%
casing                     80.60%       58.20%
keyboard                   83.82%       81.09%
spelling                   88.91%       87.20%
abbreviations              90.78%       89.31%

──────────────────────────────────────────────────────────────────────
OOS Recall per Noise Type (on OOS test queries)
──────────────────────────────────────────────────────────────────────
Noise                  Linear OOS    Proto OOS
──────────────────────────────────────────────────────────────────────
original                    0.00%       39.90%
casing                      0.00%       86.00%
keyboard                    0.00%       